# 173 — Prediction Formation Tracking

Track how predictions form step by step: timeline of top predictions,
commitment point, prediction drivers, alternative trajectories,
and stability analysis.

## Why This Matters

Prediction formation tracking analyzes how the model arrives at its output predictions. Tracking prediction formation across layers reveals the computational process — when the model commits to an answer, what changes its mind, and how confident it is.

**Key references:**
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.prediction_formation_tracking import (
    prediction_timeline,
    commitment_point,
    prediction_drivers,
    alternative_prediction_analysis,
    prediction_stability,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
result = prediction_timeline(model, tokens, top_k=5)
for s in result['stages']:
    tops = ', '.join(f"tok{t['token']}({t['prob']:.3f})" for t in s['top_predictions'])
    print(f"{s['stage']:8s}: [{tops}]")

In [ ]:
result = commitment_point(model, tokens)
print(f"Final token: {result['final_token']}  Commit layer: {result['commit_layer']}\n")
for t in result['trajectory']:
    status = 'COMMITTED' if t['is_committed'] else f'pred={t["top_token"]}'
    print(f"  Layer {t['layer']}: {status}  conf={t['final_token_confidence']:.4f}")

In [ ]:
result = prediction_drivers(model, tokens)
print(f"Target: token {result['target_token']}\n")
for d in result['drivers']:
    bar = '#' * int(d['abs_logit'] * 10)
    sign = '+' if d['logit'] > 0 else '-'
    print(f"  {d['component']:10s}: {sign}{d['abs_logit']:.4f}  {bar}")

In [ ]:
result = alternative_prediction_analysis(model, tokens, top_k=3)
for tok in result['per_token']:
    ranks = ', '.join(f"L{t['layer']}@{t['rank']}" for t in tok['trajectory'])
    print(f"Token {tok['token']}: final_rank={tok['final_rank']}  [{ranks}]")

In [ ]:
result = prediction_stability(model, tokens)
print(f"Stability: {result['stability']:.3f}  Changes: {result['n_changes']}  "
      f"Longest streak: {result['longest_streak']}")
print(f"Predictions: {result['predictions']}")